# 02 — Baseline Models

Before training scGPT, establish baselines that any model must beat.

**Why baselines matter:** Recent papers (Nature Methods 2025) showed that deep learning models don't always beat simple linear baselines on perturbation prediction. Knowing your baselines lets you check whether your model is actually learning something.

**Two baselines:**
1. `MeanControl` — predict the control mean for every perturbation (zero-effect model)
2. `DeltaMean` — predict the average observed perturbation effect (Train Mean baseline)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gears import PertData

from src.baselines import MeanControlBaseline, DeltaMeanBaseline
from src.evaluate import format_baseline_results, compute_metrics, filter_ctrl_from_results

## 1. Load data (reuse from notebook 01)

In [ ]:
pert_data = PertData('../data')
pert_data.load(data_name='adamson')
pert_data.prepare_split(split='simulation', seed=1)
pert_data.get_dataloader(batch_size=64, test_batch_size=64)

adata = pert_data.adata
ctrl_adata = adata[adata.obs['condition'] == 'ctrl']
ctrl_mean = np.asarray(ctrl_adata.X.mean(axis=0)).flatten()

# set2conditions gives {'train': [...], 'val': [...], 'test': [...]}
# Note: 'ctrl' is already included in set2conditions['train']
train_conditions = pert_data.set2conditions['train']
train_adata = adata[adata.obs['condition'].isin(train_conditions)]
test_perts = pert_data.set2conditions['test']

print(f"Train perts: {len(pert_data.set2conditions['train'])}")
print(f"Test perts: {len(test_perts)}")

## 2. MeanControl baseline

**Hypothesis:** Predict the control mean expression for every perturbation.

This is the zero-effect model — it predicts that nothing changes. It will score near-perfect on overall Pearson (most genes *don't* change), but zero on `pearson_de_delta` (it predicts no perturbation effect).

Use this as your floor: any model that doesn't beat this on `pearson_de_delta` is useless.

In [ ]:
ctrl_baseline = MeanControlBaseline()
ctrl_baseline.fit(ctrl_adata)

ctrl_preds = ctrl_baseline.predict(test_perts)
ctrl_results = format_baseline_results(ctrl_preds, pert_data)

ctrl_metrics = compute_metrics(filter_ctrl_from_results(ctrl_results), ctrl_adata)
print('MeanControl metrics:')
for k, v in ctrl_metrics.items():
    print(f'  {k}: {v:.4f}')

## 3. DeltaMean baseline

**Hypothesis:** For test perturbations, predict ctrl_mean + (average delta across all training perturbations).

This is the competitive baseline. It captures the global direction of perturbation effects without knowing anything specific about the test genes.

In [ ]:
delta_baseline = DeltaMeanBaseline()
delta_baseline.fit(train_adata, ctrl_adata)

delta_preds = delta_baseline.predict(test_perts)
delta_results = format_baseline_results(delta_preds, pert_data)

delta_metrics = compute_metrics(filter_ctrl_from_results(delta_results), ctrl_adata)
print('DeltaMean metrics:')
for k, v in delta_metrics.items():
    print(f'  {k}: {v:.4f}')

## 4. Compare and visualize

**Key insight:** Overall Pearson will be high for both — this is misleading. Focus on `pearson_delta` and `pearson_de`.

In [ ]:
results_table = pd.DataFrame({
    'MeanControl': ctrl_metrics,
    'DeltaMean': delta_metrics,
}).T

print('Baseline comparison:')
print(results_table.to_string(float_format='{:.4f}'.format))
print()
print('Key metric: pearson_de_delta (delta from ctrl on top-20 DE genes)')
print('What to expect from scGPT (Adamson dataset):')
print('  scGPT pearson_de_delta: ~0.80')
print('  DeltaMean pearson_de_delta: should be clearly lower')

In [ ]:
# Visualize predictions vs truth for one test perturbation
test_pert = test_perts[0]
true_expr = np.asarray(adata[adata.obs['condition'] == test_pert].X.mean(axis=0)).flatten()
ctrl_pred_expr = ctrl_preds[test_pert]
delta_pred_expr = delta_preds[test_pert]

delta = true_expr - ctrl_mean
top_de_idx = np.argsort(np.abs(delta))[-50:]  # top 50 differentially expressed genes

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (pred_name, pred_expr) in zip(axes, [
    ('MeanControl', ctrl_pred_expr), 
    ('DeltaMean', delta_pred_expr)
]):
    ax.scatter(true_expr[top_de_idx], pred_expr[top_de_idx], alpha=0.6, s=20)
    ax.plot([true_expr[top_de_idx].min(), true_expr[top_de_idx].max()],
            [true_expr[top_de_idx].min(), true_expr[top_de_idx].max()],
            'r--', label='perfect prediction')
    r = np.corrcoef(true_expr[top_de_idx], pred_expr[top_de_idx])[0, 1]
    ax.set_title(f'{pred_name}\nPearson (top 50 DE) = {r:.3f}\nPerturbation: {test_pert}')
    ax.set_xlabel('True expression')
    ax.set_ylabel('Predicted expression')
    ax.legend()

plt.tight_layout()
plt.show()

## 5. Save baseline metrics

We'll load these in notebook 04 to compare against scGPT.

In [ ]:
import json
import os
os.makedirs('../outputs', exist_ok=True)

baseline_metrics = {
    'MeanControl': ctrl_metrics,
    'DeltaMean': delta_metrics,
}
with open('../outputs/baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2)

print('Saved to outputs/baseline_metrics.json')

delta_de_delta = delta_metrics.get('pearson_de_delta', 0)
ctrl_de_delta = ctrl_metrics.get('pearson_de_delta', 0)

# MeanControl should score near-zero on pearson_de_delta (predicts no change)
assert ctrl_de_delta < 0.20, f'MeanControl pearson_de_delta too high: {ctrl_de_delta:.3f}'
# DeltaMean should score meaningfully positive
assert delta_de_delta > 0.20, f'DeltaMean pearson_de_delta too low: {delta_de_delta:.3f}'
# DeltaMean should beat MeanControl
assert delta_de_delta > ctrl_de_delta, 'DeltaMean should beat MeanControl on pearson_de_delta'

print(f'Phase 3 validation passed!')
print(f'  DeltaMean pearson_de_delta = {delta_de_delta:.4f}')
print(f'  MeanControl pearson_de_delta = {ctrl_de_delta:.4f}')
print(f'  scGPT target: pearson_de_delta ~0.80')

In [ ]:
delta_de_score = delta_metrics.get('pearson_de', 0)
ctrl_de_score = ctrl_metrics.get('pearson_de', 1)

assert delta_de_score > 0.60, f'DeltaMean pearson_de too low: {delta_de_score:.3f}'
assert ctrl_de_score < 0.30, f'MeanControl should be near-zero on pearson_de: {ctrl_de_score:.3f}'
assert delta_de_score > ctrl_de_score, 'DeltaMean should beat MeanControl'

print(f'Phase 3 validation passed!')
print(f'  DeltaMean pearson_de = {delta_de_score:.4f} (target: >0.60)')
print(f'  MeanControl pearson_de = {ctrl_de_score:.4f} (target: <0.30)')
print(f'  scGPT target: pearson_de ~0.80')